## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [68]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re
import math


## Setting Up the Device

In [69]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [70]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [71]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 1000
alt_model = False

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [72]:
# This is Cell #5

vocab: list[str] = []
for c in sequence:
    if c not in vocab:
        vocab.append(c)

char_to_idx = {c: i for i, c in enumerate(vocab)}
idx_to_char = {v: k for k, v in char_to_idx.items()}

data = [char_to_idx[c] for c in sequence]


## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [73]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [74]:
# This is Cell #7

if not alt_model: # Alphabet model
    sequence_length = 32     # Length of each input sequence
    stride = 10              # Stride for creating sequences
    embedding_dim = 16       # Dimension of character embeddings
    hidden_size = 16         # Number of features in the hidden state of the RNN
    learning_rate = 0.5      # Learning rate for the optimizer
    num_epochs = 10          # Number of epochs to train
    batch_size = 64          # Batch size for training
    vocab_size = len(vocab)
    input_size = len(vocab)
    output_size = len(vocab)
else: # War and Peace model
    sequence_length = 64    # Length of each input sequence
    stride = 10              # Stride for creating sequences
    embedding_dim = 128      # Dimension of character embeddings
    hidden_size = 256        # Number of features in the hidden state of the RNN
    learning_rate = 0.1     # Learning rate for the optimizer
    num_epochs = 10          # Number of epochs to train
    batch_size = 64          # Batch size for training
    vocab_size = len(vocab)
    input_size = len(vocab)
    output_size = len(vocab)


After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

* **sequence_length**
    * The length of the window of characters the RNN will process as a sequence
    * For the alphabet dataset: I set this to 32 because the training data is highly patterned and repeats every 26 characters, so I chose a number a bit above that
    * For the *War and Peace* dataset: I set this to 64 because this is around the length of a sentence in characters, which should be good for a piece of literature like this
* **stride**
    * The number of characters between each sequence the RNN will process
    * For the alphabet dataset: I set this to 10 because to capture the z to a wrap-around, the stride needs to be shorter than 26 to allow for this and gives more chance for the model to learn the alphabet
    * For the *War and Peace* dataset: I set this to 10 to capture good overlap between parts of sentences
* **embedding_dim**
    * The number of dimensions the character embedding vectors output by the embedding layer have
    * For the alphabet dataset: I set this to 16 because the embeddings should carry good information about the characters and how they relate to other characters, but since the data is so repetitive, 26 would be overkill
    * For the *War and Peace* dataset: I set this to 128 to capture the relationships of different characters to each other and represent the more complex interdependence they have in this dataset
* **hidden_size**
    * The number of dimensions the hidden vector that keeps track of state has
    * For the alphabet dataset: I set this to 16 because the repetitiveness of the training data is so high, the model should not need much context from previous characters to be able to predict the next well
    * For the *War and Peace* dataset: I set this to 256 to keep track of a nice chunk of information from previous characters and their relationships to better represent the complex interdependence of words and sentence structure
* **learning_rate**
    * The rate at which the optimizer will adjust weights during gradient descent
    * For the alphabet dataset: I set this to 0.5 because though the rate should be small for stable learning, since the training data is small and repetitive, I gave it a high learning rate to get to a stable state quickly knowing that the precision doesn't need to be so high as the pattern should be "obvious" to the model
    * For the *War and Peace* dataset: I set this to 0.1, which is pretty high, but worked for letting the model learn these complex relationships quickly
* **num_epochs**
    * The number of times the training set is processed by the model
    * For the alphabet dataset: I set this to 10 because though the model will not need to take long to learn the pattern because it already repeats 100 times in one go, 10 should be enough to make sure it gets a chance to converge to the point that it has very, very low loss.
    * For the *War and Peace* dataset: I set this to 10 to let the model get to a pretty good convergence as it settled into diminishing gains
* **batch_size**
    * The size of the batches the RNN will process before adjusting weights during gradient descent
    * For the alphabet dataset: I set this to 64 because it is pretty standard and doesn't need to be adjusted much in this scenario because it is not going to really introduce bias in this dataset
    * For the *War and Peace* dataset: I set this to 64 for the same reasons as the alphabet dataset
* **vocab_size**
    * The number of unique tokens in the sequences the RNN can process
    * This is simply the number of tokens in vocab
* **input_size**
    * The number of dimensions in the input vectors
    * This will be the same as vocab_size for one-hot encodings
* **output_size**
    * The number of dimensions in the output vector

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [75]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)
train_size = math.floor(0.9 * len(data_tensor))
train_data = data_tensor[:train_size]
test_data = data_tensor[train_size:]


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [76]:
# This is Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, drop_last=True, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, drop_last=True, shuffle=False)
print(len(train_dataset), len(test_dataset))

total_batches = len(train_loader)


2337 257


## Defining the RNN Model

Here we will define our character-level RNN model.

In [77]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [78]:
# This is Cell #12

model = CharRNN(input_size, hidden_size, output_size, embedding_dim)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [79]:
# This is Cell #13
for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/10:   0%|          | 0/36 [00:00<?, ?it/s]C:\Users\ezlot\AppData\Local\Temp\ipykernel_4744\1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
C:\Users\ezlot\AppData\Local\Temp\ipykernel_4744\1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/10: 100%|██████████| 36/36 [00:00<00:00, 113.92it/s]


Epoch [1/10], Loss: 1.7400, Accuracy: 85.07%


Epoch 2/10: 100%|██████████| 36/36 [00:00<00:00, 117.44it/s]


Epoch [2/10], Loss: 0.2858, Accuracy: 99.94%


Epoch 3/10: 100%|██████████| 36/36 [00:00<00:00, 114.66it/s]


...

Epoch [8/10], Loss: 0.0261, Accuracy: 100.00%


Epoch 9/10: 100%|██████████| 36/36 [00:00<00:00, 132.57it/s]


Epoch [9/10], Loss: 0.0225, Accuracy: 100.00%


Epoch 10/10: 100%|██████████| 36/36 [00:00<00:00, 109.75it/s]

Epoch [10/10], Loss: 0.0197, Accuracy: 100.00%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [80]:
# This is Cell #14
sequence = read_file('warandpeace.txt')
alt_model = True

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [81]:
# This is Cell #15
model.eval()

with torch.no_grad():
    total_loss, correct_predictions, total_predictions = 0, 0, 0
    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Testing"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        # Forward pass
        output, hidden = model(batch_inputs, hidden)
        
        # Detach hidden state to avoid carrying gradients
        hidden = hidden.detach()

        # Calculate loss
        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets

        # Calculate accuracy
        _, predicted_indices = torch.max(output, dim=2)  # Get predicted characters
        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    # Calculate average loss and accuracy
    avg_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage

    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

Testing:   0%|          | 0/4 [00:00<?, ?it/s]C:\Users\ezlot\AppData\Local\Temp\ipykernel_4744\1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
C:\Users\ezlot\AppData\Local\Temp\ipykernel_4744\1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Testing: 100%|██████████| 4/4 [00:00<00:00, 307.70it/s]

Test Loss: 0.0183, Test Accuracy: 100.00%


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [82]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()
    
    input_sequence = torch.tensor([char_to_idx[char] for char in start_text], dtype=torch.long).unsqueeze(0).to(device)  # [1, len(start_sequence)]

    generated_text = start_text
    hidden = model.init_hidden(batch_size=1)

    for _ in range(k):
        with torch.no_grad():
            # Forward pass through the model
            logits, hidden = model(input_sequence, hidden)
            last_logits = logits[:, -1, :]

            # Get the prediction for the next character
            next_char_idx = int(sample_from_output(last_logits, temperature))
            next_char = idx_to_char[next_char_idx]
            generated_text += next_char

            # Prepare the next input sequence
            input_sequence = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)
    
    return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
Exiting...


## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.


Answer:  
I observed very low loss values for the alphabet dataset, reducing to less than 0.3 in only two epochs, with a final loss of 0.0208 and 100% accuracy. On the test data, it scored a 100% accuracy with 0.0197 loss. Accuracy was at or near 100% for the most of the epochs of training as well. When testing generated text with both low and high temperatures, the model perfectly continued alphabetical strings, only outputting inaccurate results when temperature was raised very high (>1). Even when given nonsense input, the model would output the alphabet starting with the last letter of the input string at temperatures lower than 1.  
Examples for the alphabet dataset:  
* Input: "qrstuv" with temperature 1 | Output: "qrstuvwxyzabcde"
* Input: "efghij" with temperature 1 | Output: "efghijklmnopqrst"
* Input: "cdefg" with temperature 2 | Output: "cdefghabcjkzast"
* Input: "fdoindfaodh" with temperature 0.5 | Output: "fdoindfaodhijklmnopqr"

The *War and Peace* dataset, on the other hand, yielded less predictible results. The final loss was higher at 1.4370 with a lower accuracy of 57.62%. On the test data, it scored a 55.81% accuracy with 1.5042 loss. This result is expected, as the model is being trained on much more complex and varied data. Unlike the alphabet dataset, *War and Peace* uses words and sentences that carry a lot of meaning, i.e. language that is much more difficult to predict. Despite the lower accuracy, the model still performed fairly well at producing legible results, though not at a level that created sentences with real meaning. At high temperatures, the model output incoherent responses and often failed to form real words, but at middle temperatures (0.2-0.5), the model almost always output real words with some degree of coherence, though the sentences themselves were still usually largely (semantically) nonsense. At low temperatures (<=0.1), the model was prone to using common words and to high repetition.
Examples for the *War and Peace* dataset:  
* Input: "i do not " with temperature 1 | Output: "i do not yet your refears calchusing the regirl. mrya love."
* Input: "there will not be " with temperature 0.4 | Output: "there will not be seen what was a fear and the staff that he had not"
* Input: "here is a " with temperature 0.1 | Output: "here is a strange and the countess and the countess were the"
  
These two models obviously performed very differently. This is largely due to the fact that the first model was trained on extremely patterned and predictible data, which is easier for a neural network to learn and generate with high accuracy, while to produce a well-trained model on a larger, much more complex and varied dataset, even with ideal hyperparameters, generalized performance is expected to be worse. The tuning for the *War and Peace* dataset was also harder for me to pinpoint without risking overfitting and massively increasing training time to learn more complex relationships between characters.

The most challenging part of this project was the tweaking of the hyperparameters to work well with the alphabet and *War and Peace* datasets as well as understanding the inner workings of the RNN model. While working on this project, I developed a firmer grasp on how RNNs use neural networks to learn transformation matrices to abstract sematic meaning into numbers and use those numbers to produce meaningful output. I also developed deeper understanding of the hyperparameters that define this process and how they impact the RNN model's ability to train on different datasets.